# 03 — Limpieza y Preparación

Fase **Data Preparation** de CRISP-DM. Generamos versiones limpias de los cuatro datasets en `data/processed/` listas para integrarse en el notebook 04.

**Decisiones clave (tomadas tras el EDA del notebook 02):**

1. **Horizonte temporal limitado:** el calendar se filtra a las **primeras 8 semanas** desde la fecha de scrape (2025-12-14 → 2026-02-08). Justificación: en Inside Airbnb `available='f'` mezcla *reservado* y *bloqueado por el anfitrión*; solo en las primeras semanas predomina la reserva real. Esto evita que el modelo aprenda ruido.
2. **Variable objetivo:** `occupied = (available == 'f').astype(int)` — binaria, clases razonablemente equilibradas.
3. **Listings:** 25 columnas útiles; se descarta `price` (100 % nulo).
4. **Imputación:** scores de reviews con mediana por barrio; `reviews_per_month` con 0; `host_is_superhost` / `instant_bookable` convertidos a binario.
5. **Clima:** `weathercode` WMO agrupado en 4 categorías (soleado / nublado / lluvia / extremo).
6. **Eventos:** agregado a `n_eventos_por_dia` con categoría dominante del día; se rellenan con 0 los días sin eventos.

## 0. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option('display.max_columns', 100)

RAW_DIR = Path('../data/raw')
PROC_DIR = Path('../data/processed')
PROC_DIR.mkdir(parents=True, exist_ok=True)

# Fecha del scrape y ventana del modelo
SCRAPE_DATE = pd.Timestamp('2025-12-14')
HORIZON_WEEKS = 8
HORIZON_END = SCRAPE_DATE + pd.Timedelta(weeks=HORIZON_WEEKS)

print(f'Scrape: {SCRAPE_DATE.date()}')
print(f'Horizonte del modelo: {SCRAPE_DATE.date()} → {HORIZON_END.date()}  ({HORIZON_WEEKS} semanas)')

Scrape: 2025-12-14
Horizonte del modelo: 2025-12-14 → 2026-02-08  (8 semanas)


---
## 1. Listings — selección e imputación

In [3]:
listings = pd.read_csv(RAW_DIR / 'listings.csv')
print(f'Original: {listings.shape}')

cols_utiles = [
    'id', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed',
    'latitude', 'longitude',
    'property_type', 'room_type',
    'accommodates', 'bathrooms', 'bedrooms', 'beds',
    'minimum_nights', 'maximum_nights',
    'availability_30', 'availability_60', 'availability_90', 'availability_365',
    'estimated_occupancy_l365d',
    'number_of_reviews', 'reviews_per_month',
    'review_scores_rating', 'review_scores_location', 'review_scores_value',
    'host_is_superhost', 'instant_bookable'
]
listings = listings[cols_utiles].copy()
print(f'Seleccionado: {listings.shape}')

Original: (18177, 85)
Seleccionado: (18177, 25)


In [4]:
# Booleanos t/f → 1/0
for c in ['host_is_superhost', 'instant_bookable']:
    listings[c] = (listings[c] == 't').astype(int)

# Imputación de reviews_per_month: 0 (sin reviews significa 0 reviews/mes)
listings['reviews_per_month'] = listings['reviews_per_month'].fillna(0)

# Imputación de scores por mediana del barrio, y si el barrio no tiene datos, mediana global
for col in ['review_scores_rating', 'review_scores_location', 'review_scores_value']:
    med_barrio = listings.groupby('neighbourhood_cleansed')[col].transform('median')
    med_global = listings[col].median()
    listings[col] = listings[col].fillna(med_barrio).fillna(med_global)

# Nulos numéricos residuales en bathrooms/bedrooms/beds: mediana por room_type
for col in ['bathrooms', 'bedrooms', 'beds']:
    med = listings.groupby('room_type')[col].transform('median')
    listings[col] = listings[col].fillna(med).fillna(listings[col].median())

print('Nulos después de imputar:')
print(listings.isna().sum()[listings.isna().sum() > 0])

Nulos después de imputar:
Series([], dtype: int64)


In [5]:
# Property_type: quedarnos con los top 10 y agrupar el resto como 'Other'
top10 = listings['property_type'].value_counts().head(10).index
listings['property_type'] = listings['property_type'].where(
    listings['property_type'].isin(top10), 'Other'
)

# Eliminar listings con datos sin sentido (bathrooms=0, accommodates=0)
n0 = len(listings)
listings = listings[(listings['accommodates'] > 0) & (listings['bedrooms'].fillna(0) <= 20)]
print(f'Descartados {n0 - len(listings)} listings con valores absurdos')

listings.to_csv(PROC_DIR / 'listings_clean.csv', index=False)
print(f'Guardado: listings_clean.csv  shape={listings.shape}')
listings.head()

Descartados 5 listings con valores absurdos
Guardado: listings_clean.csv  shape=(18172, 25)


,id,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,availability_30,availability_60,availability_90,availability_365,estimated_occupancy_l365d,number_of_reviews,reviews_per_month,review_scores_rating,review_scores_location,review_scores_value,host_is_superhost,instant_bookable
0,18674,la Sagrada Família,Eixample,41.40556,2.17262,Entire rental unit,Entire home/apt,8,2.0,3.0,6.0,1,1125,20,50,74,115,54,54,0.35,4.36,4.81,4.34,0,1
1,2031134,el Fort Pienc,Eixample,41.40001,2.18416,Entire rental unit,Entire home/apt,4,1.0,2.0,3.0,2,1125,17,42,68,294,255,453,3.16,4.78,4.79,4.71,1,1
2,4415694,el Poble Sec,Sants-Montjuïc,41.37230,2.16441,Entire rental unit,Entire home/apt,5,1.0,2.0,2.0,3,1125,23,46,69,115,66,182,1.39,4.38,4.67,4.20,0,0
3,4415780,la Vila de Gràcia,Gràcia,41.40667,2.16152,Private room in rental unit,Private room,2,1.0,1.0,1.0,1,3,0,0,0,0,0,5,0.04,4.40,4.75,4.50,0,0
4,5064035,la Vila de Gràcia,Gràcia,41.39996,2.16339,Entire rental unit,Entire home/apt,6,1.5,2.0,4.0,3,1125,4,20,39,222,138,128,0.99,4.41,4.73,4.26,0,1


---
## 2. Calendar — filtrado al horizonte + variable objetivo

In [6]:
calendar = pd.read_csv(
    RAW_DIR / 'calendar.csv',
    parse_dates=['date'],
    dtype={'available': 'category', 'listing_id': 'int64'},
    usecols=['listing_id','date','available','minimum_nights','maximum_nights']
)
print(f'Original: {calendar.shape}')

# Filtrar al horizonte del modelo
calendar = calendar[(calendar['date'] >= SCRAPE_DATE) & (calendar['date'] < HORIZON_END)].copy()
print(f'Tras filtrar a {HORIZON_WEEKS} semanas: {calendar.shape}')

# Solo listings que siguen en listings_clean
calendar = calendar[calendar['listing_id'].isin(listings['id'])].copy()
print(f'Tras intersectar con listings_clean: {calendar.shape}')

Original: (6634623, 5)
Tras filtrar a 8 semanas: (1013063, 5)
Tras intersectar con listings_clean: (1012783, 5)


In [7]:
# Variable objetivo
calendar['occupied'] = (calendar['available'] == 'f').astype(int)

# Features temporales
calendar['dow']       = calendar['date'].dt.dayofweek       # 0=Lun
calendar['is_weekend']= (calendar['dow'] >= 5).astype(int)
calendar['month']     = calendar['date'].dt.month
calendar['day']       = calendar['date'].dt.day
calendar['week']      = calendar['date'].dt.isocalendar().week.astype(int)

# Festivos de Cataluña en el rango (diciembre 2025 → febrero 2026)
festivos = pd.to_datetime([
    '2025-12-25', '2025-12-26',   # Navidad, Sant Esteve
    '2026-01-01', '2026-01-06',   # Año Nuevo, Reyes
])
calendar['is_holiday'] = calendar['date'].isin(festivos).astype(int)

# Limpiamos available (ya tenemos occupied)
calendar = calendar.drop(columns=['available'])

# Tasa de ocupación resultante con el nuevo filtro
print(f'\n% ocupado en el horizonte: {calendar["occupied"].mean()*100:.2f}%')
print(f'Filas: {len(calendar):,}')
print(f'Listings: {calendar["listing_id"].nunique():,}')
print(f'Días: {calendar["date"].nunique()}')
calendar.head()


% ocupado en el horizonte: 54.17%
Filas: 1,012,783
Listings: 18,172
Días: 56


,listing_id,date,minimum_nights,maximum_nights,occupied,dow,is_weekend,month,day,week,is_holiday
0,660131950555612074,2025-12-14,1,999,1,6,1,12,14,50,0
1,660131950555612074,2025-12-15,1,999,0,0,0,12,15,51,0
2,660131950555612074,2025-12-16,1,999,0,1,0,12,16,51,0
3,660131950555612074,2025-12-17,1,999,0,2,0,12,17,51,0
4,660131950555612074,2025-12-18,1,999,0,3,0,12,18,51,0


In [8]:
calendar.to_csv(PROC_DIR / 'calendar_clean.csv', index=False)
print(f'Guardado: calendar_clean.csv  shape={calendar.shape}')

Guardado: calendar_clean.csv  shape=(1012783, 11)


---
## 3. Clima — agrupación de `weathercode`

In [9]:
clima = pd.read_csv(RAW_DIR / 'clima_barcelona.csv', parse_dates=['date'])
print(f'Original: {clima.shape}')

# Filtrar al horizonte del modelo
clima = clima[(clima['date'] >= SCRAPE_DATE) & (clima['date'] < HORIZON_END)].copy()
print(f'Tras filtrar al horizonte: {clima.shape}')

# Agrupar WMO weathercode (https://open-meteo.com/en/docs)
def categorize_weather(code):
    if code in (0, 1):
        return 'soleado'
    elif code in (2, 3):
        return 'nublado'
    elif code in (45, 48, 51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82):
        return 'lluvia'
    else:  # 71+, 85, 86, 95, 96, 99 - tormentas, nieve, niebla densa
        return 'extremo'

clima['weather_cat'] = clima['weathercode'].apply(categorize_weather).astype('category')
clima['temp_avg'] = (clima['temperature_2m_max'] + clima['temperature_2m_min']) / 2
clima['has_rain'] = (clima['precipitation_sum'] > 0.1).astype(int)

print('\nDistribución de weather_cat:')
print(clima['weather_cat'].value_counts())

clima.to_csv(PROC_DIR / 'clima_clean.csv', index=False)
print(f'\nGuardado: clima_clean.csv  shape={clima.shape}')
clima.head()

Original: (396, 7)
Tras filtrar al horizonte: (56, 7)

Distribución de weather_cat:
weather_cat
lluvia     35
nublado    20
extremo     1
Name: count, dtype: int64

Guardado: clima_clean.csv  shape=(56, 10)


,date,temperature_2m_max,temperature_2m_min,precipitation_sum,windspeed_10m_max,weathercode,fuente,weather_cat,temp_avg,has_rain
13,2025-12-14,14.8,4.9,2.3,11.9,55,observado,lluvia,9.85,1
14,2025-12-15,15.1,10.3,36.6,14.4,65,observado,lluvia,12.70,1
15,2025-12-16,14.8,8.0,2.0,14.3,53,observado,lluvia,11.40,1
16,2025-12-17,15.8,8.5,0.1,12.0,51,observado,lluvia,12.15,0
17,2025-12-18,15.2,6.9,0.0,9.2,3,observado,nublado,11.05,0


---
## 4. Eventos — agregación diaria (dataset auxiliar, no se integra en el modelo)

**Limitación confirmada:** Ticketmaster solo sirve eventos futuros desde el momento de la llamada; el horizonte del modelo (2025-12-14 → 2026-02-08) ya ha pasado, así que no hay solapamiento posible. La variable `n_eventos` quedaría a 0 en todas las filas y no aportaría información.

**Decisión:** se limpian y guardan igualmente (`eventos_clean.csv`) para subirlos a BigQuery como tabla auxiliar de BI, pero **no se integran** en `dataset_integrado.csv` ni en el modelo de ocupación. Se documenta esta limitación en la memoria.

La agregación se hace sobre el rango original de eventos (abril-julio 2026), no sobre el horizonte del modelo.

In [10]:
eventos = pd.read_csv(RAW_DIR / 'eventos_barcelona.csv', parse_dates=['date'])
print(f'Original: {eventos.shape}')
print(f'Rango de eventos: {eventos["date"].min().date()} → {eventos["date"].max().date()}')
print(f'Horizonte del modelo : {SCRAPE_DATE.date()} → {(HORIZON_END - pd.Timedelta(days=1)).date()}')
print(f'Solapamiento con el horizonte: {((eventos["date"] >= SCRAPE_DATE) & (eventos["date"] < HORIZON_END)).sum()} eventos')

# Agregar por día (rango completo de eventos, no filtramos al horizonte del modelo)
eventos_diarios = (eventos.groupby('date')
                          .agg(n_eventos=('id','count'),
                               categoria_top=('category',
                                              lambda s: s.mode().iat[0] if not s.mode().empty else 'ninguno'))
                          .reset_index())

print(f'\nDías con al menos 1 evento: {len(eventos_diarios)}')
print(f'Media eventos/día: {eventos_diarios["n_eventos"].mean():.1f}')
print(f'Max eventos/día: {eventos_diarios["n_eventos"].max()}')

eventos_diarios.to_csv(PROC_DIR / 'eventos_clean.csv', index=False)
print(f'\nGuardado: eventos_clean.csv  shape={eventos_diarios.shape}')
print('ATENCIÓN: este fichero NO se integra en el modelo (limitación de la API Ticketmaster).')
eventos_diarios.head()

Original: (1000, 8)
Rango de eventos: 2026-04-22 → 2026-07-09
Horizonte del modelo : 2025-12-14 → 2026-02-07
Solapamiento con el horizonte: 0 eventos

Días con al menos 1 evento: 70
Media eventos/día: 14.3
Max eventos/día: 22

Guardado: eventos_clean.csv  shape=(70, 3)
ATENCIÓN: este fichero NO se integra en el modelo (limitación de la API Ticketmaster).


,date,n_eventos,categoria_top
0,2026-04-22,11,Miscellaneous
1,2026-04-23,12,Miscellaneous
2,2026-04-24,13,Miscellaneous
3,2026-04-25,19,Miscellaneous
4,2026-04-26,20,Miscellaneous


---
## 5. Resumen de ficheros limpios

In [11]:
print('=' * 60)
print('FICHEROS LIMPIOS EN data/processed/')
print('=' * 60)

resumen = []
for f in sorted(PROC_DIR.glob('*.csv')):
    df = pd.read_csv(f, nrows=5)
    total_rows = sum(1 for _ in open(f)) - 1
    size_mb = f.stat().st_size / (1024 * 1024)
    resumen.append({
        'fichero': f.name,
        'filas': f'{total_rows:,}',
        'cols': len(df.columns),
        'size_mb': f'{size_mb:.1f}',
    })
    print(f'\n{f.name}  ({size_mb:.1f} MB, {total_rows:,} filas, {len(df.columns)} cols)')
    print(f'  Columnas: {list(df.columns)}')

pd.DataFrame(resumen)

FICHEROS LIMPIOS EN data/processed/

calendar_clean.csv  (46.0 MB, 1,012,783 filas, 11 cols)
  Columnas: ['listing_id', 'date', 'minimum_nights', 'maximum_nights', 'occupied', 'dow', 'is_weekend', 'month', 'day', 'week', 'is_holiday']

clima_clean.csv  (0.0 MB, 56 filas, 10 cols)
  Columnas: ['date', 'temperature_2m_max', 'temperature_2m_min', 'precipitation_sum', 'windspeed_10m_max', 'weathercode', 'fuente', 'weather_cat', 'temp_avg', 'has_rain']

eventos_clean.csv  (0.0 MB, 70 filas, 3 cols)
  Columnas: ['date', 'n_eventos', 'categoria_top']

listings_clean.csv  (2.9 MB, 18,172 filas, 25 cols)
  Columnas: ['id', 'neighbourhood_cleansed', 'neighbourhood_group_cleansed', 'latitude', 'longitude', 'property_type', 'room_type', 'accommodates', 'bathrooms', 'bedrooms', 'beds', 'minimum_nights', 'maximum_nights', 'availability_30', 'availability_60', 'availability_90', 'availability_365', 'estimated_occupancy_l365d', 'number_of_reviews', 'reviews_per_month', 'review_scores_rating', 'review_

,fichero,filas,cols,size_mb
0,calendar_clean.csv,"1,012,783",11,46.0
1,clima_clean.csv,56,10,0.0
2,eventos_clean.csv,70,3,0.0
3,listings_clean.csv,"18,172",25,2.9


---
## 6. Siguiente paso — notebook 04 (integración)

Con los tres CSVs limpios principales en `data/processed/`, el notebook 04 hará el join en granularidad **listing × día**:

```
dataset_integrado.csv  =  calendar_clean
                           ⟕ listings_clean  (por listing_id)
                           ⟕ clima_clean     (por date)
```

`eventos_clean.csv` se sube a BigQuery como tabla auxiliar (`datos.eventos_barcelona`) pero no entra en el join principal por la limitación temporal de Ticketmaster documentada arriba.

El CSV resultante tendrá ~1 M filas × ~32 columnas, listo para:
- Subir a **BigQuery** como `datos.fact_ocupacion`.
- Importar en **Orange Data Mining** para entrenar los clasificadores.
- Conectar desde **Power BI** al proyecto de BigQuery para el dashboard final.